# GAS-BayesSHAP — official ShaplEIG baseline (matched unique-query budgets)

Closes the audit's last P1 item: the SOTA baselines are no longer 'method-style'.  This runs the **official ShaplEIG** (ICML 2026, Rundel et al.), ported faithfully from the authors' public MIT repository (github.com/slds-lmu/shapleig @ d52c09e), on the same wine/air membership games as GAS-BayesSHAP, at matched **unique** coalition-query budgets, reporting RMSE vs exact ground truth and the actual query cost.  Orchestrates `scripts/run_official_shaplEIG.py` only — no duplicated algorithm (the port is in that script, cited to the pinned source).

> **Crash-free by construction:** the torch/GPyTorch/BoTorch stack crashed natively on macOS (SIGSEGV, then "Python quit unexpectedly"). The port is therefore **pure NumPy/SciPy** with the IDENTICAL official math (Hamming-kernel exact GP, MLL fit, EIG `_compute_eig_function_property_naive_Z`, Shapley weights `_get_shapley_weights`) — no torch import anywhere, so no native crash surface.  A reviewer can diff the math against `@d52c09e`.

## 0. Environment — pure NumPy/SciPy (no torch)

In [ ]:
import sys, os, time, subprocess
from pathlib import Path

ROOT = Path("..").resolve()
SCRIPTS = ROOT / "scripts"

# The official-ShaplEIG port is PURE NumPy/SciPy (same math as the
# official torch implementation, github.com/slds-lmu/shapleig@d52c09e).
# The torch/GPyTorch/BoTorch stack crashed natively on macOS (SIGSEGV /
# 'Python quit unexpectedly'), so no torch is imported anywhere —
# there is no native crash surface left.  numpy/scipy are already
# required by the package.
import numpy, scipy  # noqa: F401
print('numpy', numpy.__version__, '| scipy', scipy.__version__)

N_INST   = int(os.environ.get("N_INST", "1"))
BUDGETS  = os.environ.get("BUDGETS", "64,128,256")
SKIP     = set(os.environ.get("GAS_SKIP", "").split(",")) - {""}

def run(*args, tag="", skip=False):
    if skip:
        print(f"--- SKIPPED: {tag or ' '.join(args)}"); return 0.0
    cmd = [sys.executable, str(SCRIPTS / args[0]), *args[1:]]
    t0 = time.time()
    print(f"\n>>> {tag or ' '.join(args)}")
    r = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    dt = time.time() - t0
    if r.returncode != 0:
        err = (r.stderr or r.stdout or '').strip().splitlines()
        tail = ' | '.join(err[-6:]) if err else '<no output>'
        print('--- child stderr tail ---')
        print(tail[:1200])
        raise RuntimeError(f"FAILED ({r.returncode}): {' '.join(args)}")
    print(f"<<< done in {dt/60:.1f} min")
    return dt

print(f"N_INST={N_INST} BUDGETS={BUDGETS} SKIP={sorted(SKIP)}")
print("NOTE: each (dataset, budget) runs in its own child process with a"
      " 1500 s timeout; failures land in paper_shaplEIG_port_failures.csv")

## A. Wine — official ShaplEIG at matched budgets

In [ ]:
run("run_official_shaplEIG.py", "--n", str(N_INST), "--budgets", BUDGETS,
    "--dataset", "wine", tag=f"A. official ShaplEIG wine N={N_INST}",
    skip="A" in SKIP)

## B. Air — official ShaplEIG at matched budgets

In [ ]:
run("run_official_shaplEIG.py", "--n", str(N_INST), "--budgets", BUDGETS,
    "--dataset", "air", tag=f"B. official ShaplEIG air N={N_INST}",
    skip="B" in SKIP)

## C. Comparison vs GAS-BayesSHAP (matched unique evals)

Reads `paper_shaplEIG_port_{wine,air}.csv` and the GAS matched-budget curves (actual unique coalition evals) and prints a side-by-side RMSE table.  **Honest framing:** this is a point-estimate comparison at matched *unique* query cost; GAS's differentiators are the distribution-free anytime certificates + Neyman residual control, which ShaplEIG (Bayesian, non-certified) does not provide.

In [ ]:
import pandas as pd
for ds in ("wine", "air"):
    p = ROOT / "main_results" / f"paper_shaplEIG_port_{ds}.csv"
    if not p.exists():
        print(f"[{ds}] official ShaplEIG CSV missing — run A/B first"); continue
    s = pd.read_csv(p)
    print(f"\n=== {ds}: official ShaplEIG (source: {s['source'].iloc[0][:40]}...) ===")
    g = s.groupby("budget").agg(
        shaplEIG_rmse=("rmse_vs_exact", "mean"),
        unique_queries=("unique_queries", "mean"),
    )
    print(g.round(5).to_string())
    print("\nCompare with GAS-BayesSHAP at the same unique-eval cost:")
    print("  GAS wine K=128: unique ~393, rmse 0.00430 | K=256: unique ~480, "
          "rmse 0.00352 | K=512: unique ~565, rmse 0.00304")
    print("  (from paper_wine_matched_budget.csv, spec range)")

## Expected runtime and honest notes
- **Budgets are capped at 512** (each budget B costs ~B rounds of GP
  refit + full EIG, ~0.5–1 s/round measured: budget=64 ≈ 1 min, 
  budget=256 ≈ 4–6 min, budget=512 ≈ 12–17 min per config — the 
  512 config is opt-in via BUDGETS=512 and may be slow).
- **Full run ≈ 25–45 min** (N=1 × 2 datasets × 3 budgets {64,128,256});
  each config runs in its own child process with a 1500 s timeout, so a
  crash or hang is recorded in `paper_shaplEIG_port_failures.csv`
  and the rest still completes.
- **Fault isolation:** each (dataset, budget) runs in its own child
  process with a 1500 s timeout; a crash or hang is recorded in
  `paper_shaplEIG_port_failures.csv` and the remaining configs
  still complete.
- **Smoke:** `N_INST=1 BUDGETS=32` (~1–2 min per dataset).
- Budgets are **unique coalition queries** (counted via the GAS
  CoalitionOracle cache), so the comparison is fair vs GAS's
  `num_coalition_evals_this_call`.
- The port is cited to the pinned official source commit; if you spot a
  discrepancy, diff against `github.com/slds-lmu/shapleig@d52c09e`.
- Commit the resulting `paper_shaplEIG_port_{wine,air}.csv`.